# ME 203 — Test 4: Iterative systems and matrix transformations

**35 minutes total for the Canvas quiz and this notebook · 22 points total**

Canvas questions: **10 points**. This notebook: **4 tasks × 3 points = 12 points**.

Name: **[Tega Ikoba]**

Suggested pacing: about 10 minutes for the quiz, 22 minutes for programming, and 3 minutes to check, save, and upload. You may divide your time differently.

- Follow the permitted-resource rules posted with the test in Canvas.
- Keep the given function and variable names.
- Calculate results; do not hard-code answers.
- Functions must work with other inputs satisfying the documented assumptions.
- Only NumPy is required. No course `.py` files, external data files, animations, Otter, or source JSON files are needed.
- Inspect cells show outputs but are not autograder checks.
- Each task is intended to be independently gradable.
- Before the timer ends, restart the kernel, run all cells, save with outputs, and upload this `.ipynb` file to the programming upload question. Leave attempted code for partial credit.


In [2]:
import numpy as np

# Provided helper functions for the matrix problems.
# Do not change these.

def make_scale_matrix(scale_x=1.0, scale_y=1.0):
    mat = np.identity(3)
    mat[0, 0] = scale_x
    mat[1, 1] = scale_y
    return mat

def make_translation_matrix(d_x=0.0, d_y=0.0):
    mat = np.identity(3)
    mat[0, 2] = d_x
    mat[1, 2] = d_y
    return mat

def make_rotation_matrix(theta=0.0):
    mat = np.identity(3)
    mat[0, 0] = np.cos(theta)
    mat[0, 1] = -np.sin(theta)
    mat[1, 0] = np.sin(theta)
    mat[1, 1] = np.cos(theta)
    return mat


## Task 1 — Predator/prey derivatives (3 points)

Complete `predator_prey_derivative`.

The current population is supplied as `[prey, predator]`. Use the equations

$
\frac{d(\text{prey})}{dt} =
a(\text{prey}) - b(\text{prey})(\text{predator})
$

$
\frac{d(\text{predator})}{dt} =
-c(\text{predator}) + d(\text{prey})(\text{predator})
$

where the parameters are stored as `"prey_reproduce"`, `"prey_eaten"`,
`"predator_loss"`, and `"predator_reproduce"`. where where the parameters are stored as `a = prey_reproduce`, `b = prey_eaten`,
`c = predator_loss`, and `d = predator_reproduce`.

Return a NumPy array containing **the two derivatives**, in that order.
Do **not** multiply by a time step in this function.


In [5]:
def predator_prey_derivative(prey_predator, params):
    prey, predator = prey_predator

    # TODO: calculate the two derivatives.
    dprey_dt = (params['prey_reproduce'] * prey) - (params['prey_eaten'] * predator)
    dpredator_dt = -(params['predator_loss'] * predator) + (params['predator_reproduce'] * prey * predator)

    return np.array([dprey_dt, dpredator_dt])


In [6]:
params_1 = {
    "prey_reproduce": 1.0,
    "prey_eaten": 0.01,
    "predator_loss": 1.2,
    "predator_reproduce": 0.02,
}
derivs = predator_prey_derivative(np.array([100.0, 40.0]), params_1)
print("Derivatives:", derivs)


Derivatives: [99.6 32. ]


## Task 2 — Iterate a system with Euler integration (3 points)

Complete `simulate_euler`.

- `initial_values` contains the starting state.
- `derivative_function(current_values, params)` returns the derivative of each state.
- Store the **initial state in row 0**.
- Perform `n_steps` Euler updates using

$
x_{next} = x_{current} + \Delta t \, \dot{x}_{current}
$

- Return an array with shape `(n_steps + 1, number_of_states)`.

Use the supplied `derivative_function`; do not hard-code the predator/prey equations into this function.


In [13]:
def simulate_euler(initial_values, params, delta_t, n_steps, derivative_function):
    # TODO: allocate the result and store the initial state.
    trajectory = np.zeros((n_steps+1, 2))
    trajectory[0] = initial_values

    # TODO: calculate and store each new state.
    x_initial = initial_values[0]
    vx_initial = initial_values[1]
    for i in range(n_steps+1):
        trajectory[i, :] = (x_initial, vx_initial)
        next_x, next_vx = (x_initial + delta_t * x_initial), (derivative_function((x_initial, vx_initial), params)[0] * delta_t + derivative_function((x_initial, vx_initial), params)[1])
        x_initial, vx_initial = next_x, next_vx
    return trajectory


In [14]:
def simple_derivative(values, params):
    # Independent test system: dx/dt = vx, dvx/dt = acceleration
    x, vx = values
    return np.array([vx, params["acceleration"]])

trajectory = simulate_euler(
    initial_values=np.array([2.0, 1.0]),
    params={"acceleration": -2.0},
    delta_t=0.5,
    n_steps=3,
    derivative_function=simple_derivative,
)

print(trajectory)
print("Shape:", trajectory.shape)


[[ 2.     1.   ]
 [ 3.    -1.5  ]
 [ 4.5   -2.75 ]
 [ 6.75  -3.375]]
Shape: (4, 2)


## Task 3 — Build the shape matrix for a link (3 points)

A basic square extends from `-1` to `1` in both x and y. Complete
`make_link_shape_matrix(link_length, link_width)` so the transformed square:

- has total length `link_length`,
- has total width `link_width`,
- has the **middle of its left edge at `(0, 0)`**, and
- extends in the positive x direction.

Use the provided `make_scale_matrix` and `make_translation_matrix`.
Remember that matrix multiplication order determines which transformation happens first.


In [ ]:
def make_link_shape_matrix(link_length, link_width):
    # TODO: build and return the 3x3 shape matrix.
    mat_link = ...
    return mat_link


In [ ]:
mat_link = make_link_shape_matrix(link_length=4.0, link_width=2.0)
print(mat_link)

# Corners of a unit square in homogeneous coordinates.
pts_square = np.array([
    [-1.0,  1.0,  1.0, -1.0],
    [-1.0, -1.0,  1.0,  1.0],
    [ 1.0,  1.0,  1.0,  1.0],
])

pts_link = mat_link @ pts_square
print("x range:", np.min(pts_link[0, :]), "to", np.max(pts_link[0, :]))
print("y range:", np.min(pts_link[1, :]), "to", np.max(pts_link[1, :]))


## Task 4 — Find the end of a two-link arm (3 points)

Complete `two_link_end_location`.

Both links start along the positive x-axis before rotations are applied.

- Link 1 has length `length_1` and rotates by `theta_1` about the origin.
- Link 2 attaches to the end of link 1 and rotates by `theta_2` **relative to link 1**.
- Use homogeneous 3×3 transformation matrices.
- Return only the final `(x, y)` location of the end of link 2 as a NumPy array.

A useful way to think about the second link pose is:

1. rotate with link 1,
2. move to the end of link 1,
3. apply link 2's relative rotation.

Do not use trigonometric end-point equations directly; build and use the transformation matrices.


In [ ]:
def two_link_end_location(length_1, length_2, theta_1, theta_2):
    # TODO: build the pose matrix for link 2.
    mat_pose_link_2 = ...

    # The end of link 2 is length_2 along its local +x axis.
    end_local = np.array([length_2, 0.0, 1.0])

    # TODO: transform the point to world coordinates.
    end_world = ...

    return end_world[0:2]


In [ ]:
end_location = two_link_end_location(
    length_1=2.0,
    length_2=1.0,
    theta_1=np.pi / 2.0,
    theta_2=-np.pi / 2.0,
)
print("End location:", end_location)


## Submission

Restart the kernel, run all cells, and save. Check that the printed outputs appear and that you have left any attempted work in place for partial credit.

Upload this notebook with its outputs to the Test 4 programming upload question **before the timer ends**. No other programming files are required.
